### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
### Read all the pdf's from the directory
def process_all_pdfs(pdf_directory):

    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} Pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf
 Loaded 1270 Pages

Processing: Personal_Questions.pdf
 Loaded 2 Pages

Total documents loaded: 1272


In [3]:
##Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", "", " "]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk\
    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [4]:
chunks = split_documents(all_pdf_documents)
chunks

Split 1272 documents into 3728 chunks

Example Chunk:
Content: FIFTH EDITION
BEHROUZ A. FOROUZAN
Data Communications
AND Networking
The ﬁ  fth edition of Behrouz Forouzan’s Data Communications and Networking 
presents a comprehensive and accessible approach to da...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2012-07-11T14:31:07+08:00', 'moddate': '2014-10-25T12:41:29+03:00', 'source': '..\\data\\pdf\\Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf', 'total_pages': 1270, 'page': 1, 'page_label': 'i', 'source_file': 'Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2012-07-11T14:31:07+08:00', 'moddate': '2014-10-25T12:41:29+03:00', 'source': '..\\data\\pdf\\Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf', 'total_pages': 1270, 'page': 1, 'page_label': 'i', 'source_file': 'Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf', 'file_type': 'pdf'}, page_content='FIFTH EDITION\nBEHROUZ A. FOROUZAN\nData Communications\nAND Networking\nThe ﬁ  fth edition of Behrouz Forouzan’s Data Communications and Networking \npresents a comprehensive and accessible approach to data communications and \nnetworking that has made this book a favorite with students and professionals alike.  \nMore than 830 ﬁ  gures and 150 tables accompany the text and provide a visual \nand intuitive opportunity for understanding the material. This unique approach \nminimizes the need for heavy math content, allowing normally complicated \ntopics to unfold grap

### Embedding and VectorStore DB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    def __init__(self, model_name : str="all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [7]:
# from torch import embedding
import os

class VectorStore:
    def __init__(self, collection_name: str="pdf_documents", persist_directory: str="../data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collectio: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise



    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Preare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)


            # Document Content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())


        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print("Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore



    

Vector store initialized. Collection: pdf_documents
Existing documents in collectio: 22344


In [8]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2012-07-11T14:31:07+08:00', 'moddate': '2014-10-25T12:41:29+03:00', 'source': '..\\data\\pdf\\Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf', 'total_pages': 1270, 'page': 1, 'page_label': 'i', 'source_file': 'Data Communications and Networking By Behrouz A. Forouzan 5th Edition.pdf', 'file_type': 'pdf'}, page_content='FIFTH EDITION\nBEHROUZ A. FOROUZAN\nData Communications\nAND Networking\nThe ﬁ  fth edition of Behrouz Forouzan’s Data Communications and Networking \npresents a comprehensive and accessible approach to data communications and \nnetworking that has made this book a favorite with students and professionals alike.  \nMore than 830 ﬁ  gures and 150 tables accompany the text and provide a visual \nand intuitive opportunity for understanding the material. This unique approach \nminimizes the need for heavy math content, allowing normally complicated \ntopics to unfold grap

In [9]:
#  Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 3728 texts...


Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Generated embeddings with shape: (3728, 384)
Adding 3728 documents to vector store...
Successfully added 3728 documents to vector store
Total documents in collection: 26072


### Retriever Pipeline From VectorStore

In [10]:
class RAGRetriever:
    def __init__(self, vector_store:VectorStore, embedding_manager:EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager



    def retrieve(self, query:str, top_k: int=5, score_threshold: float=0.0) -> List[Dict[str, Any]]:


        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshould: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                    
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever


In [21]:
rag_retriever.retrieve("What is data communication?")

Retrieving documents for query: 'What is data communication?'
Top K: 5, Score threshould: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3b55f0eb_81',
  'content': '4 PART I OVERVIEW\n1.1 DATA COMMUNICATIONS\nWhen we communicate, we are sharing info rmation. This sharing can be local or\nremote. Between individuals, local communica tion usually occurs face to face, while\nremote communication takes place over distance. The term telecommunication, which\nincludes telephony, telegraphy, and television, means communication at a distance (tele\nis Greek for “far”). The word data refers to information pres ented in whatever form is\nagreed upon by the parties creating and using the data. \nData communications  are the exchange of data between two devices via some\nform of transmission medium such as a wi re cable. For data communications to occur,\nthe communicating devices must be part of a communication system made up of a com-\nbination of hardware (physical equipment) and software (programs). The effectiveness\nof a data communications system depends on four fundamental characteristics: deliv-\nery, accuracy

In [20]:
rag_retriever.retrieve("who are you?")

Retrieving documents for query: 'who are you?'
Top K: 5, Score threshould: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)


[]

### Integration Vectordb Context pipeline Wit LLM output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

#  Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    # Generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer  the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [14]:
answer = rag_simple("who is ahmar jadran? ", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'who is ahmar jadran? '
Top K: 3, Score threshould: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


### Enhanced RAD Pipeline Features

In [18]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context = False):

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence':0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate Answer
    prompt = f"Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output


# Example usage:
result = rag_advanced("what is your nme?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print('Sources:', result['sources'])
print('Confidence:', result['confidence'])
print('Context Preview:', result['context'][:300])

Retrieving documents for query: 'what is your nme?'
Top K: 3, Score threshould: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [17]:
# Advance RAG Pipeline: streaming, Citations, History, Summarization....
from typing import Dict, List, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []


    def query(self, question:str, top_k: int=5, min_score: float=0.2, stream: bool=False, summarize: bool=False ) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context= "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'], 
                'preview': doc['content'][:120] + '...'
            }for doc in results]
            prompt = f"Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
            if stream:
                print('Streaming Answer:')
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add Citations to answer:
        citations = [f"[{i+1}] {src['source']} (page{src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer
    
        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary

        })
        return { 
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary, 
            'history': self.history

        }

# Example Usage
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is your name?", top_k=3, min_score=0.1, stream=True, summarize = True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result["summary"])
print("History:", result['history'][-1])




Retrieving documents for query: 'what is your name?'
Top K: 3, Score threshould: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)
Retrieved 0 documents (after filtering)

Final Answer: No relevant context found.
Summary: There is no information to summarize as the provided answer lacks context.
History: {'question': 'what is your name?', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'There is no information to summarize as the provided answer lacks context.'}
